# Week 2 Assessment — Autonomous Deep Research Agent

**Author: Jemine Mene-Ejegi**

Enhanced version of `4_lab4.ipynb` (Deep Research) with every requirement from `captions.txt`:

| Enhancement | What it does | Source requirement |
|---|---|---|
| **Clarifying Questions Agent** | Before searching, asks 3 targeted questions to sharpen the query | *"come up with three clarifying questions... incorporate those in what searches are done"* |
| **Truly Agentic Manager** | An actual `Agent` (not a Python script) that autonomously decides when to search more, refine queries, and hand off to the writer | *"turn that manager from just being a Python script into something that's truly Agentic... agents as tools and handoffs"* |
| **Autonomous Query Refinement** | Manager re-plans and re-searches based on what it has already learned, up to 3 rounds | *"give it some ability to go off and explore and also potentially to refine the queries"* |
| **Evaluator-Optimizer Pattern** | A dedicated evaluator agent grades the report; score < 6 triggers a targeted follow-up research round | *"Evaluator-Optimizer design patterns, so there's another agent that gets to check the work"* |
| **Gradio UI** | Two-phase chat interface that surfaces clarifying questions before running research | *"fiddling around with Gradio to surface those questions"* |

In [ ]:
from agents import Agent, Runner, trace, gen_trace_id, WebSearchTool, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from IPython.display import display, Markdown
import gradio as gr
import sendgrid
import asyncio
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict

In [ ]:
load_dotenv(override=True)

openai_key = os.getenv('OPENAI_API_KEY')
sendgrid_key = os.getenv('SENDGRID_API_KEY')

print(f"OpenAI key  : {'found (' + openai_key[:8] + '...)' if openai_key else 'MISSING'}")
print(f"SendGrid key: {'found' if sendgrid_key else 'MISSING — emails will be skipped'}")

## Architecture

```
User enters query
      │
      ▼
Clarifying Questions Agent  →  3 questions shown in Gradio UI
      │
User answers questions
      │
      ▼
Research Manager Agent   ←── truly agentic: LLM controls loop
  ├─ tool: plan_searches   (planner_agent.as_tool)
  ├─ tool: search_web      (search_agent.as_tool)
  │     └─ loops up to 3 rounds; refines queries from findings
  └─ handoff ───────────────────────────► Writer Agent
                                                    │  output_type=ReportData
                                                    ▼
                                           Evaluator Agent
                                             ├─ score ≥ 6 → Email Agent → send
                                             └─ score < 6 → Research Manager (follow-up)
                                                                  └─ Writer → Email Agent
```

---
## 1 — Clarifying Questions Agent

Mirrors how OpenAI's own Deep Research product works: before any web search happens, the agent asks 3 targeted questions to understand the user's intent, desired depth, and constraints.

In [ ]:
class ClarifyingQuestions(BaseModel):
    questions: list[str] = Field(
        description="Exactly 3 clarifying questions to sharpen the research query."
    )


clarify_agent = Agent(
    name="Clarifying Questions Agent",
    instructions=(
        "You help users sharpen their research queries by generating exactly 3 targeted clarifying questions. "
        "Your questions must uncover: "
        "(1) the specific angle or perspective the user wants, "
        "(2) the desired depth and target audience (technical vs high-level overview), "
        "(3) any constraints, time-frame, or focus area. "
        "Keep each question concise and directly actionable for directing web searches."
    ),
    output_type=ClarifyingQuestions,
    model="gpt-5.4-mini",
)

print("Clarifying Questions Agent ready.")

---
## 2 — Search and Planner Agents

Same agents as `4_lab4.ipynb`, but here they are exposed as **tools** for the Research Manager, giving the manager LLM full autonomous control over when and what to search.

In [ ]:
HOW_MANY_SEARCHES = 4  # searches generated per planning round (manager can do up to 3 rounds)

search_agent = Agent(
    name="Search Agent",
    instructions=(
        "You are a research assistant. Given a search term, search the web and produce a concise summary "
        "of the results in 2-3 paragraphs, under 300 words. Capture the main points succinctly. "
        "This will be consumed by a researcher synthesising a report — capture the essence and ignore fluff. "
        "Do not include any commentary beyond the summary itself."
    ),
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-5.4-mini",
    model_settings=ModelSettings(tool_choice="required"),
)


class WebSearchItem(BaseModel):
    reason: str = Field(description="Why this search is important to the query.")
    query: str = Field(description="The search term to use.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform.")


planner_agent = Agent(
    name="Planner Agent",
    instructions=(
        f"You are a research planning assistant. Given a query (and any user clarifications), "
        f"generate {HOW_MANY_SEARCHES} distinct, targeted web search terms that together will give "
        f"comprehensive coverage of the topic from multiple angles."
    ),
    output_type=WebSearchPlan,
    model="gpt-5.4-mini",
)

print(f"Search Agent and Planner Agent ready ({HOW_MANY_SEARCHES} searches per planning round).")

---
## 3 — Writer Agent

Receives all gathered search results from the Research Manager via handoff and synthesises them into a structured, detailed report with structured output.

In [ ]:
class ReportData(BaseModel):
    short_summary: str = Field(description="A 2-3 sentence executive summary of the findings.")
    markdown_report: str = Field(description="The full report in markdown, at least 1000 words.")
    follow_up_questions: list[str] = Field(description="5 suggested follow-up research questions.")


writer_agent = Agent(
    name="Writer Agent",
    instructions=(
        "You are a senior researcher writing a cohesive, detailed report. "
        "You will receive the original query, user clarifications, and all summarised search results. "
        "Create an outline first, then write the full report in markdown format, at least 1000 words, "
        "with clear sections. Synthesise across all sources rather than just listing them. "
        "Be analytical, insightful, and appropriate for a knowledgeable professional audience."
    ),
    output_type=ReportData,
    model="gpt-5.4-mini",
)

print("Writer Agent ready.")

---
## 4 — Evaluator Agent (Evaluator-Optimizer Pattern)

After the writer produces a report, this agent grades it across 5 dimensions (comprehensiveness, depth, accuracy, structure, relevance). If the score is below 6/10, the pipeline automatically triggers a targeted follow-up research round.

In [ ]:
class ReportEvaluation(BaseModel):
    is_acceptable: bool = Field(
        description="True if the report is comprehensive and ready to send (quality_score >= 6)."
    )
    quality_score: int = Field(
        ge=1, le=10,
        description="Quality score from 1 (very poor) to 10 (excellent)."
    )
    feedback: str = Field(
        description="Specific, actionable feedback — what is good and what needs improvement."
    )
    missing_topics: list[str] = Field(
        description="Topics or angles missing or underdeveloped in the report."
    )


evaluator_agent = Agent(
    name="Report Evaluator",
    instructions=(
        "You are a rigorous quality evaluator for deep research reports. "
        "Evaluate the report on five dimensions: "
        "(1) Comprehensiveness — multiple angles covered; "
        "(2) Depth — beyond surface-level; "
        "(3) Accuracy — claims are well-supported and consistent; "
        "(4) Structure — well-organised with clear sections; "
        "(5) Relevance — directly and fully addresses the original query. "
        "Be strict but fair. Award 7+ only for genuinely high-quality, insightful reports. "
        "Set is_acceptable=True if quality_score >= 6. Flag all obvious gaps in missing_topics."
    ),
    output_type=ReportEvaluation,
    model="gpt-5.4-mini",
)

print("Evaluator Agent ready.")

---
## 5 — Email Agent

Converts the finished report to clean HTML and sends it via SendGrid.

In [ ]:
SENDER_EMAIL = "jemineme@gmail.com"      # verified SendGrid sender
RECIPIENT_EMAIL = "jaymineh94@gmail.com"  # where the report is sent


@function_tool
def send_report_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send the research report as a formatted HTML email via SendGrid."""
    if not sendgrid_key:
        print("[EMAIL] SendGrid key not set — skipping email.")
        return {"status": "skipped", "reason": "no SENDGRID_API_KEY"}
    sg = sendgrid.SendGridAPIClient(api_key=sendgrid_key)
    mail = Mail(
        Email(SENDER_EMAIL),
        To(RECIPIENT_EMAIL),
        subject,
        Content("text/html", html_body),
    ).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}


email_agent = Agent(
    name="Email Agent",
    instructions=(
        "You are an email formatter and sender. You receive a markdown research report. "
        "Convert it to clean, professional HTML with a readable layout. "
        "Generate an appropriate subject line that reflects the research topic. "
        "Then send it using the send_report_email tool."
    ),
    tools=[send_report_email],
    model="gpt-5.4-mini",
)

print(f"Email Agent ready. Sender: {SENDER_EMAIL} → {RECIPIENT_EMAIL}")

---
## 6 — Research Manager Agent (Truly Agentic)

**This is the key difference from `4_lab4.ipynb`.** The original lab uses a Python class `ResearchManager` with hardcoded steps. Here, the manager is an actual `Agent` — the LLM itself decides:

- How many rounds of searching to run
- Whether to refine queries based on what it has already found  
- When it has enough information to hand off to the writer

This makes it agentic under Anthropic's definition: *"a system in which an LLM controls the workflow"*.

In [ ]:
# Expose planner and search agents as tools the manager LLM can call autonomously
plan_searches_tool = planner_agent.as_tool(
    tool_name="plan_searches",
    tool_description=(
        f"Given a research topic (and any user clarifications or gaps identified from previous searches), "
        f"generate {HOW_MANY_SEARCHES} distinct, targeted web search queries. "
        "Returns a structured plan. Call this at the start of each research round."
    ),
)

search_web_tool = search_agent.as_tool(
    tool_name="search_web",
    tool_description=(
        "Execute a single web search for the given query and return a concise summary. "
        "Call this once per search term from your plan. "
        "Input should be: 'Search term: <term>\\nReason for searching: <reason>'"
    ),
)


MANAGER_INSTRUCTIONS = """
You are an autonomous deep research manager. You directly control the research workflow.
The quality of the final report depends entirely on the decisions YOU make.

YOUR PROCESS:

ROUND 1 - Initial research:
1. Call `plan_searches` with the full research query and user clarifications.
2. Call `search_web` for EACH item in the plan (one call per search term).
3. Review all results: Are there important angles not covered? Conflicting claims to verify?
   Interesting threads worth following up?

ROUND 2 (highly recommended if any gaps exist):
4. Call `plan_searches` again with a refined focus targeting the gaps you identified.
5. Call `search_web` for each new search term.

ROUND 3 (optional, only if critical gaps remain):
6. One final targeted round to fill remaining gaps.

HAND OFF:
7. When satisfied (typically 8-12 searches across multiple angles), hand off to the writer agent.
   Your handoff message MUST include:
   - The original query and user clarifications
   - ALL search results verbatim, clearly labelled with their search term

RULES:
- You MUST use the handoff to the writer agent. Do not write the report yourself.
- Pursue interesting threads. Aim for both breadth AND depth.
- Maximum 3 planning rounds to control costs.
- Always run at least 2 planning rounds unless the first round already yields rich results.
"""

research_manager_agent = Agent(
    name="Research Manager",
    instructions=MANAGER_INSTRUCTIONS,
    tools=[plan_searches_tool, search_web_tool],
    handoffs=[writer_agent],
    model="gpt-5.4-mini",
)

print("Research Manager Agent ready.")
print("  Tools    : plan_searches, search_web")
print("  Handoffs : Writer Agent")
print("  Model    : gpt-5.4-mini")

---
## 7 — Research Pipeline

Python function that orchestrates the full flow:
1. Research Manager runs autonomously → hands off to Writer
2. Evaluator grades the report
3. If score < 6: targeted follow-up round + re-write
4. Email Agent sends the final report

In [ ]:
async def run_research(query: str, clarifications: str) -> tuple:
    """
    Full pipeline:
      Research Manager (agentic) -> Writer -> Evaluator -> (optional follow-up) -> Email
    Returns (ReportData, ReportEvaluation, trace_url).
    """
    combined_input = (
        f"Research Query: {query}\n\n"
        f"User Clarifications:\n{clarifications}"
    )

    trace_id = gen_trace_id()
    trace_url = f"https://platform.openai.com/traces/trace?trace_id={trace_id}"
    print(f"\nTrace: {trace_url}")

    with trace("Deep Research (Autonomous)", trace_id=trace_id):

        # Step 1: Research Manager -> Writer (via autonomous handoff)
        print("[1/4] Research Manager running autonomously...")
        result = await Runner.run(
            research_manager_agent,
            combined_input,
            max_turns=40,
        )
        report = result.final_output_as(ReportData)
        print(f"      Report written: {len(report.markdown_report):,} chars")

        # Step 2: Evaluate
        print("[2/4] Evaluating report quality...")
        eval_result = await Runner.run(
            evaluator_agent,
            f"Original query: {query}\nUser clarifications: {clarifications}\n\nReport:\n{report.markdown_report}"
        )
        evaluation = eval_result.final_output_as(ReportEvaluation)
        print(f"      Score: {evaluation.quality_score}/10 | acceptable: {evaluation.is_acceptable}")

        # Step 3: Follow-up if quality is insufficient
        if not evaluation.is_acceptable:
            gaps = ", ".join(evaluation.missing_topics) if evaluation.missing_topics else "general depth"
            print(f"[3/4] Score {evaluation.quality_score}/10 — running follow-up research for: {gaps}")
            follow_up = (
                f"Research Query: {query}\n"
                f"User Clarifications: {clarifications}\n\n"
                f"CONTEXT: A first-draft report was produced but an evaluator found it lacking.\n"
                f"Evaluator feedback: {evaluation.feedback}\n"
                f"Missing/underdeveloped topics: {gaps}\n\n"
                f"Please do targeted follow-up searches to address these gaps, then produce an improved report."
            )
            result2 = await Runner.run(research_manager_agent, follow_up, max_turns=25)
            report = result2.final_output_as(ReportData)
            print(f"      Improved report: {len(report.markdown_report):,} chars")
        else:
            print("[3/4] Quality check passed — no follow-up needed.")

        # Step 4: Send email
        print("[4/4] Sending email...")
        await Runner.run(email_agent, report.markdown_report)
        print("      Done!")

    return report, evaluation, trace_url


print("Research pipeline ready.")

---
## 8 — Gradio UI

Two-phase chat:
- **Turn 1**: User types the query → Clarifying Questions Agent returns 3 questions
- **Turn 2**: User answers the questions → full autonomous research pipeline runs → report displayed
- **Turn 3+**: User is prompted to clear the chat for a new topic

In [ ]:
async def research_chat(message: str, history: list) -> str:
    """
    Gradio ChatInterface async callback.
    Turn 0 (no history) -> generate clarifying questions.
    Turn 1 (1 prior exchange) -> run full research pipeline.
    Turn 2+ -> ask user to start a new session.
    """
    user_turns = [m for m in history if m["role"] == "user"]

    # Phase 1: Generate clarifying questions
    if len(user_turns) == 0:
        print(f"[QUERY] {message}")
        result = await Runner.run(clarify_agent, message)
        questions = result.final_output
        resp = "Before I start researching, I need a few clarifications:\n\n"
        for i, q in enumerate(questions.questions, 1):
            resp += f"**{i}. {q}**\n\n"
        resp += (
            "Please answer all three questions in your next message. "
            "Even brief answers help me target the research much more effectively. "
            "I'll then run the autonomous research and email you the final report."
        )
        return resp

    # Phase 2: Run research with clarifications
    elif len(user_turns) == 1:
        original_query = user_turns[0]["content"]
        clarifications = message
        print(f"[RESEARCH] query='{original_query}'")
        try:
            report, evaluation, trace_url = await run_research(original_query, clarifications)
        except Exception as e:
            return f"\u274c Research failed with error: {e}"

        resp = "## \u2705 Research Complete\n\n"
        resp += f"[Trace] {trace_url}\n\n"
        resp += f"**Quality Score: {evaluation.quality_score}/10**"
        if not evaluation.is_acceptable:
            resp += " (follow-up round was automatically run)"
        resp += "\n\n"
        resp += f"**Executive Summary:** {report.short_summary}\n\n"
        resp += f"---\n\n{report.markdown_report}\n\n"
        resp += "---\n\n### Suggested Follow-up Questions\n\n"
        resp += "\n".join([f"- {q}" for q in report.follow_up_questions])
        resp += f"\n\n---\n\n*Evaluator: {evaluation.feedback}*"
        resp += "\n\nReport emailed to you!"
        return resp

    # Phase 3: Session complete
    else:
        return (
            "This research session is complete. To research a new topic, "
            "clear the chat history (trash icon) or refresh the page and start again."
        )


print("Gradio callback ready.")

In [ ]:
ui = gr.ChatInterface(
    fn=research_chat,
    type="messages",
    title="Autonomous Deep Research Agent",
    description=(
        "Enter a research query. I will:\n"
        "1. Ask you 3 clarifying questions\n"
        "2. Autonomously plan and execute searches (refining queries along the way)\n"
        "3. Evaluate the report quality and run a follow-up round if needed\n"
        "4. Email you the final report\n\n"
        "\u26a0\ufe0f Uses OpenAI WebSearchTool (~$0.025/search \u00d7 8-12 searches ≈ $0.20-$0.30 per run)."
    ),
    examples=[
        "Latest AI agent frameworks and tools in 2026",
        "DevOps and platform engineering trends for 2026",
        "Kubernetes service mesh best practices in production",
    ],
    theme=gr.themes.Soft(primary_hue="sky"),
)

ui.launch()